In [3]:
from __future__ import annotations
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Dict, List, Protocol, ClassVar, Type
from uuid import UUID, uuid4

# ---------- Basistypen ---------- #

class HasUID(Protocol):
    uid: UUID        # sorgt für ein gemeinsames Attribut aller Knoten
    name: str

@dataclass(slots=True, frozen=True, kw_only=True)
class BaseNode:
    name: str
    uid: UUID = field(default_factory=uuid4)

    # generische Serialisierung
    def asdict(self) -> Dict:
        return asdict(self)

# ---------- konkrete Domänenobjekte ---------- #

@dataclass(slots=True, frozen=True, kw_only=True)
class Requirements(BaseNode):
    spec_file: Path | None = None

@dataclass(slots=True, frozen=True)
class Part(BaseNode):
    material: str = "UNDEF"
    geometry: Path | None = None
    model: str | None = None

@dataclass(slots=True, frozen=True, kw_only=True)
class Geometry(BaseNode):
    file: Path
    kind: str = "step"

# ---------- Beziehung ---------- #

@dataclass(slots=True, frozen=True, kw_only=True)
class Edge:
    src: UUID
    dst: UUID
    role: str

# ---------- Graph-Container ---------- #

class DataGraph:
    _registry: ClassVar[Dict[UUID, BaseNode]] = {}
    _edges: ClassVar[List[Edge]] = []

    # generisch – funktioniert für **alle** Node-Subklassen
    @classmethod
    def add(cls, node: BaseNode) -> UUID:
        cls._registry[node.uid] = node
        return node.uid

    @classmethod
    def link(cls, src: HasUID, dst: HasUID, role: str) -> None:
        cls._edges.append(Edge(src=src.uid, dst=dst.uid, role=role))

    # Beispiel-Query
    @classmethod
    def children(cls, parent: HasUID, role: str | None = None) -> List[BaseNode]:
        uids = [
            e.dst for e in cls._edges
            if e.src == parent.uid and (role is None or e.role == role)
        ]
        return [cls._registry[uid] for uid in uids]

# ---------- Demo ---------- #

req = Requirements(name="Lastenheft")
part = Part(name="Part 123", material="PP-GF30-PCR50")
geom = Geometry(name="Geom 0123", file=Path("geometry.step"))

DataGraph.add(req)
DataGraph.add(part)
DataGraph.add(geom)

DataGraph.link(part, geom, role="has_geometry")

print(DataGraph.children(part))     # -> [Geometry(...)]


[Geometry(name='Geom 0123', uid=UUID('0027743e-b2e9-4a6d-8538-07b60177eb8d'), file=PosixPath('geometry.step'), kind='step')]
